# Live News Monitor Test

This notebook tests the real IBKR news listener.

Modes:

- `ALL_NEWS`: subscribe BroadTape for all provider codes.
- `PORTFOLIO`: subscribe news for the portfolio stock list.
- `BOTH`: run BroadTape plus portfolio fallback.

By default it does not connect to IB. Set `RUN_LIVE_MONITOR = True` when TWS / IB Gateway is ready.

Note: realtime subscriptions must use source codes like `DJ`, not article channel codes like `DJ-N`.


## 0. Parameters


In [ ]:
from pathlib import Path
import os
import sqlite3
import sys
import tempfile
import time
from datetime import datetime

cwd = Path.cwd()
package_dir = cwd if cwd.name == "news_api" else cwd / "news_api"
sys.path.insert(0, str(package_dir.parent))

import importlib
import news_api.config as config_module
import news_api.portfolio_watchlist as portfolio_watchlist_module

importlib.reload(config_module)
importlib.reload(portfolio_watchlist_module)

from news_api.config import DEFAULT_REALTIME_PROVIDER_CODES, SETTINGS, NewsSettings, split_provider_codes
from news_api.portfolio_watchlist import ALL_NEWS_WATCHLIST, PORTFOLIO_WATCHLIST

# Realtime reqMktData news uses source codes, not article channel codes.
# Valid examples from IB error: BRFG, BRFUPDN, DJ, DJNL, BZ, DJTOP, FLY.
NEWS_PROVIDERS = DEFAULT_REALTIME_PROVIDER_CODES

# Choose: "ALL_NEWS", "PORTFOLIO", or "BOTH"
MODE = "BOTH"

RUN_LIVE_MONITOR = False
WAIT_SECONDS = 300
PRINT_EVERY_SECONDS = 10

# If True, every received headline will try reqNewsArticle().
FETCH_ARTICLE = False

# If True, every received headline will be pushed to Bark.
# First test with False to avoid noisy pushes.
PUSH_TO_BARK = False

LIVE_DB = Path(tempfile.mkdtemp()) / f"live_news_{MODE.lower()}.sqlite"
print("package_dir =", package_dir)
print("provider_codes =", split_provider_codes(NEWS_PROVIDERS))
print("LIVE_DB =", LIVE_DB)


## 1. Watchlist


In [ ]:
if MODE == "ALL_NEWS":
    watchlist = dict(ALL_NEWS_WATCHLIST)
elif MODE == "PORTFOLIO":
    watchlist = dict(PORTFOLIO_WATCHLIST)
elif MODE == "BOTH":
    watchlist = {**ALL_NEWS_WATCHLIST, **PORTFOLIO_WATCHLIST}
else:
    raise ValueError(f"unknown MODE: {MODE}")

print("MODE =", MODE)
print("symbols =", list(watchlist))


## 2. Output helpers


In [ ]:
def fetch_rows(db_path, limit=20):
    if not Path(db_path).exists():
        return []
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT
                r.received_at,
                r.published_at,
                r.symbol,
                r.provider,
                r.article_id,
                r.publisher,
                r.headline,
                COALESCE(a.fetch_status, '') AS fetch_status,
                COALESCE(a.error, '') AS article_error,
                COALESCE(a.article_text, '') AS article_text,
                COALESCE(e.event_type, '') AS event_type,
                COALESCE(e.importance_score, '') AS importance_score,
                COALESCE(e.should_push, '') AS should_push,
                COALESCE(p.push_status, '') AS push_status,
                COALESCE(p.response, '') AS push_response
            FROM news_raw r
            LEFT JOIN news_articles a ON a.unique_key = r.unique_key
            LEFT JOIN news_events e ON e.unique_key = r.unique_key
            LEFT JOIN news_push_log p ON p.unique_key = r.unique_key
            ORDER BY r.received_at DESC
            LIMIT ?
            """,
            (limit,),
        ).fetchall()
    return [dict(row) for row in rows]


def print_news(rows):
    if not rows:
        print("No news received yet.")
        return
    for i, row in enumerate(rows, 1):
        body = (row.get("article_text") or "").replace("\n", " ").strip()
        if len(body) > 500:
            body = body[:500] + "..."
        print("=" * 100)
        print(f"#{i} {row['received_at']} (ib_time={row.get('published_at', '')}) [{row['symbol']}] {row['provider']} {row['publisher']}")
        print("headline:", row["headline"])
        print("article:", row["fetch_status"] or "<not requested>", row["article_error"] or "")
        print("article_text:", body or "<empty>")
        print("event/score/push:", row["event_type"], row["importance_score"], "should_push=", row["should_push"], "push_status=", row["push_status"])
        if row.get("push_response"):
            print("push_response:", row["push_response"][:300])


## 3. Start live monitor


In [ ]:
if RUN_LIVE_MONITOR:
    import importlib
    import news_api.ib_client as ib_client_module
    import news_api.subscription_manager as subscription_manager_module
    import news_api.service as service_module
    import news_api.storage as storage_module
    import news_api.bark_client as bark_client_module

    importlib.reload(ib_client_module)
    importlib.reload(subscription_manager_module)
    importlib.reload(service_module)
    importlib.reload(storage_module)
    importlib.reload(bark_client_module)

    from news_api.bark_client import BarkClient
    from news_api.ib_client import IBArticleFetcher, IBNewsClient
    from news_api.service import NewsService
    from news_api.storage import SQLiteNewsStorage
    from news_api.subscription_manager import SubscriptionManager

    bark_key = os.getenv("BARK_KEY", "")
    if PUSH_TO_BARK:
        assert bark_key, "PUSH_TO_BARK=True requires BARK_KEY."

    article_fetch_score = 0 if FETCH_ARTICLE else 101
    push_score = 0 if PUSH_TO_BARK else 101

    live_settings = NewsSettings(
        host=SETTINGS.host,
        port=SETTINGS.port,
        client_id=SETTINGS.client_id,
        provider_codes=NEWS_PROVIDERS,
        db_path=LIVE_DB,
        article_fetch_score=article_fetch_score,
        portfolio_push_score=push_score,
        default_push_score=push_score,
        bark_key=bark_key,
        bark_base_url=SETTINGS.bark_base_url,
        dashboard_url=SETTINGS.dashboard_url,
    )

    storage = SQLiteNewsStorage(LIVE_DB)
    service = NewsService(
        settings=live_settings,
        watchlist=watchlist,
        storage=storage,
        bark_client=BarkClient(
            key=bark_key,
            base_url=live_settings.bark_base_url,
            dashboard_url=live_settings.dashboard_url,
            timeout=30,
            retries=2,
        ),
    )
    client = IBNewsClient(service)
    service.article_fetcher = IBArticleFetcher(client, timeout=20)

    print("Connecting IB...")
    client.start_api(live_settings.host, live_settings.port, live_settings.client_id)
    print("IB ready. next_order_id=", client.next_order_id)
    manager = SubscriptionManager(client)

    client.reqNewsProviders()
    time.sleep(2)
    print("visible_news_providers:", client.news_providers)
    print("note: visible_news_providers are article/provider channels; realtime reqMktData uses source codes.")
    realtime_provider_codes = split_provider_codes(NEWS_PROVIDERS)
    print("realtime_provider_codes:", realtime_provider_codes)

    if MODE in {"ALL_NEWS", "BOTH"}:
        broadtape_subscribed = {}
        for provider in realtime_provider_codes:
            contract_symbol = f"{provider}:{provider}_ALL"
            ticker_id = manager.subscribe_broadtape(
                provider,
                symbol_alias="ALL",
                contract_symbol=contract_symbol,
            )
            broadtape_subscribed[provider] = ticker_id
        print("BroadTape subscriptions sent:", broadtape_subscribed)

    if MODE in {"PORTFOLIO", "BOTH"}:
        subscribed = manager.subscribe_watchlist(PORTFOLIO_WATCHLIST, "+".join(realtime_provider_codes))
        print("Portfolio subscriptions sent:", subscribed)

    deadline = time.time() + WAIT_SECONDS
    try:
        while time.time() < deadline:
            now = datetime.now().strftime("%H:%M:%S")
            print("\n" + "#" * 40, now, "#" * 40)
            if client.errors:
                print("IB errors:", client.errors[-10:])
            print_news(fetch_rows(LIVE_DB, limit=10))
            time.sleep(PRINT_EVERY_SECONDS)
    finally:
        print("Stopping monitor and disconnecting IB.")
        client.stop_api()
        print("database:", LIVE_DB)
        print_news(fetch_rows(LIVE_DB, limit=50))
else:
    print("RUN_LIVE_MONITOR=False. Set it to True when IB is ready.")


## 4. Review database after monitor stops


In [ ]:
print("database:", LIVE_DB)
print_news(fetch_rows(LIVE_DB, limit=50))
